# Train an SVM classifier on the extracted features

Using the per-video HDF5 features from the previous milestone, we:

1. **Split the videos** (not the feature vectors) into 80% training / 20% testing, **stratified** so
   80% of the deepfakes *and* 80% of the originals land in the training set.
2. **Collect training features**: loop the training videos' HDF5 files and stack all their frame
   feature vectors into one array `X_train`, with a matching integer label array `y_train`
   (**0 = deepfake, 1 = original**).
3. **Train a scikit-learn SVM** on `(X_train, y_train)`.

> **Why split videos, not features?** Each video contributes many frame feature vectors. At evaluation
> time we need **one score per test video**, so *all* of a video's frames must stay together on the same
> side of the split. Mixing a video's frames across train and test would leak information and inflate
> the score.

## Collect one entry per video, with its class label

In [1]:
import glob
import numpy as np
import h5py
from pathlib import Path

root = Path.cwd()
while not (root / "Data").exists() and root != root.parent:
    root = root.parent
FEATURES_ROOT = root / "Data" / "Features"

# label convention: 1 = original/real, 0 = deepfake/fake
video_paths, video_labels = [], []
for cls, label in [("real", 1), ("fake", 0)]:
    for p in sorted(glob.glob(str(FEATURES_ROOT / cls / "**" / "*.h5"), recursive=True)):
        video_paths.append(p)
        video_labels.append(label)
video_paths  = np.array(video_paths)
video_labels = np.array(video_labels)

print("Total videos:", len(video_paths))
print("  original (1):", int((video_labels == 1).sum()))
print("  deepfake (0):", int((video_labels == 0).sum()))

Total videos: 750
  original (1): 430
  deepfake (0): 320


## Split the videos 80 / 20 (stratified by class)

`stratify=video_labels` keeps the real/fake ratio identical in both splits, which guarantees ~80% of
each class ends up in training. The split is at the **video** level.

In [2]:
from sklearn.model_selection import train_test_split

train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths, video_labels, test_size=0.2, stratify=video_labels, random_state=42)

print("Train videos: %d  (real %d, fake %d)" %
      (len(train_paths), (train_labels == 1).sum(), (train_labels == 0).sum()))
print("Test  videos: %d  (real %d, fake %d)" %
      (len(test_paths),  (test_labels  == 1).sum(), (test_labels  == 0).sum()))

Train videos: 600  (real 344, fake 256)
Test  videos: 150  (real 86, fake 64)


### Persist the split for the next milestone

Evaluation must use the **test videos only**, so we save the split. (The split is also reproducible via
`random_state=42`.)

In [3]:
MODELS_DIR = root / "Data" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

np.savez(MODELS_DIR / "video_split.npz",
         train_paths=train_paths, train_labels=train_labels,
         test_paths=test_paths,   test_labels=test_labels)
print("Saved split ->", MODELS_DIR / "video_split.npz")

Saved split -> /Users/blake/AI Projects/Deepfake-Detection/Data/models/video_split.npz


## Build the training feature & label arrays

Loop the **training** videos' HDF5 files, stack every frame's feature vector into `X_train`, and build
`y_train` of the same length (each frame inherits its video's label).

In [4]:
def collect_features(paths, labels):
    """Stack all frame feature vectors from the given videos into (X, y)."""
    X_parts, y_parts = [], []
    for path, label in zip(paths, labels):
        with h5py.File(path, "r") as h:
            feats = h["features"][:]           # (n_frames, n_features)
        X_parts.append(feats)
        y_parts.append(np.full(len(feats), label, dtype=int))
    return np.vstack(X_parts), np.concatenate(y_parts)

X_train, y_train = collect_features(train_paths, train_labels)
print("X_train:", X_train.shape, "  y_train:", y_train.shape)
print("  original frames (1):", int((y_train == 1).sum()))
print("  deepfake frames (0):", int((y_train == 0).sum()))

X_train: (6000, 35)   y_train: (6000,)
  original frames (1): 3440
  deepfake frames (0): 2560


## Train the SVM

We wrap the SVM in a pipeline with a `StandardScaler` — SVMs are sensitive to feature scale, and our
features (SSIM ≈ 1, PSNR ≈ 50, tiny histogram values) live on very different ranges.
`probability=True` lets us output per-video scores during evaluation.

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

classifier = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42),
)
classifier.fit(X_train, y_train)

print("Training done.")
print("Frame-level training accuracy: %.4f" % classifier.score(X_train, y_train))

Training done.
Frame-level training accuracy: 0.9990


### Save the trained classifier for the next milestone

In [6]:
import joblib

model_path = MODELS_DIR / "svm_deepfake.joblib"
joblib.dump(classifier, model_path)
print("Saved trained SVM ->", model_path)

Saved trained SVM -> /Users/blake/AI Projects/Deepfake-Detection/Data/models/svm_deepfake.joblib


## Summary

- Videos were split **80/20 at the video level, stratified** by class, so no video's frames straddle the
  train/test boundary and both classes keep their proportions.
- Training features were gathered from the **training videos only** into `X_train` (frame feature
  vectors) with matching integer labels `y_train` (**0 = deepfake, 1 = original**).
- A **scaled RBF SVM** was trained and saved to `Data/models/svm_deepfake.joblib`, alongside the
  train/test split.

Next milestone: load this classifier and evaluate it on the **held-out test videos**, producing one
prediction score per test video.